# 00 · Transfer NEMO to Local
Syncs raw 4i acquisition data from the Crick SMB share to local storage (`/mnt/DATA3`).
Skips files that are already present and identical (via `filecmp`). Errors are logged to `file_transfer_log.txt`.

The second section handles a one-off flat copy for the incomplete Live-2 transfer, where the SMB share was
mounted directly via `/mnt/OPERA` rather than through GVFS.

In [ ]:
import os
import shutil
import filecmp
import logging
from tqdm.auto import tqdm

## 1 · Recursive sync from SMB share
Walks the full source tree, skipping files that already exist and are identical.

In [ ]:
logging.basicConfig(
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='file_transfer_log.txt'
)

source_directory      = (
    '/run/user/30046150/gvfs/smb-share:server=data2.thecrick.org,'
    'share=lab-gutierrezm/home/shared/Shared - Baptiste Pradel/4i/Live-cell to 4i for Nathan'
)
destination_directory = '/mnt/DATA3/BPP0050'

if not os.path.exists(source_directory):
    raise FileNotFoundError(f"Source not found: '{source_directory}'")
os.makedirs(destination_directory, exist_ok=True)

total_files = sum(len(files) for _, _, files in os.walk(source_directory))

copied_files = 0
with tqdm(total=total_files, unit='file') as pbar:
    for root, _, files in os.walk(source_directory):
        for file in files:
            src  = os.path.join(root, file)
            dst  = os.path.join(destination_directory, os.path.relpath(src, source_directory))
            os.makedirs(os.path.dirname(dst), exist_ok=True)

            if not os.path.exists(dst) or not filecmp.cmp(src, dst, shallow=False):
                try:
                    shutil.copy2(src, dst)
                    copied_files += 1
                except Exception as e:
                    msg = f"Error copying '{src}': {e}"
                    print(msg)
                    logging.error(msg)
            pbar.update(1)

print(f"Transfer complete. {copied_files} files copied.")

## 2 · Flat copy for incomplete Live-2 transfer
Live-2 Images were partially missing after the recursive sync above. This section copies the flat `Images/` directory directly from the OPERA mount.

In [ ]:
def copy_images_with_progress(src_dir, dst_dir):
    """
    Copies all files from src_dir into dst_dir with a progress bar.
    Non-recursive: operates on the flat Images directory only.
    Errors are collected and printed at the end rather than interrupting the copy.

    Args:
        src_dir (str): Source Images directory path.
        dst_dir (str): Destination Images directory path.
    """
    os.makedirs(dst_dir, exist_ok=True)
    files_to_copy = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]
    total_files   = len(files_to_copy)
    copied_count  = 0
    errors        = []

    with tqdm(total=total_files, desc='Copying Images') as pbar:
        for filename in files_to_copy:
            src_path = os.path.join(src_dir, filename)
            dst_path = os.path.join(dst_dir, filename)
            try:
                shutil.copy2(src_path, dst_path)
                copied_count += 1
            except Exception as e:
                errors.append(f"Error copying '{filename}': {e}")
            pbar.update(1)

    print(f"Successfully copied {copied_count}/{total_files} files.")
    if errors:
        print("\nErrors encountered:")
        for error in errors:
            print(error)


source_directory      = (
    '/mnt/OPERA/Baptiste_Live-cell to4i for Nathan/'
    'BPP0050-1-Live-cell-to4i_Live-2__2025-04-10T18_45_48-Measurement 1/Images'
)
destination_directory = (
    '/mnt/DATA3/BPP0050/'
    'BPP0050-1-Live-cell-to4i_Live-2__2025-04-10T18_45_48-Measurement 1/acquisition/Images'
)

copy_images_with_progress(source_directory, destination_directory)